In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

# **SQL database agent**

In [6]:
from langchain_community.utilities.sql_database import SQLDatabase

sql_db = SQLDatabase.from_uri("sqlite:///Sales/sales.db")

In [9]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db= sql_db, llm= llm_openai)

In [ ]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x112724a50>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x112724a50>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x112724a50>),
 QuerySQLCheckerTool(description='Use this tool to double check if your 

In [14]:
from langchain.agents import create_agent


agent = create_agent(llm_openai, toolkit.get_tools())

In [16]:
example_query = "How much sales we made for Laptop?"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

ow much sales we made for Laptop?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_624Mzrkv0fgRCXHHgVyys311)
 Call ID: call_624Mzrkv0fgRCXHHgVyys311
  Args:
    tool_input:
================================= Tool Message =================================
Name: sql_db_list_tables

Orders
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_uH0rw8Ovpi3oUnn4pUwTahkM)
 Call ID: call_uH0rw8Ovpi3oUnn4pUwTahkM
  Args:
    table_names: Orders
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Orders" (
	id INTEGER, 
	customer_name TEXT NOT NULL, 
	product_name TEXT NOT NULL, 
	quantity INTEGER NOT NULL, 
	price REAL NOT NULL, 
	total REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from Orders table:
id	customer_name	product_name